# Neo4j Graph RAG example

This notebook creates a small Neo4j graph from a manufacturing incident document and links document chunks to graph nodes.

In [ ]:
import os
from neo4j import GraphDatabase


In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    raise ValueError("Set NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD in the environment.")


In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()


In [ ]:
document = """
Apex Motion's Pune plant manufactures BMS Controller X1.

BMS Controller X1 requires MCU-900.

MCU-900 is supplied by Shenzhen Microelectronics.

Shenzhen Microelectronics depends on Taiwan Silicon Corp
for semiconductor wafers.

Taiwan Silicon Corp operates a facility in Hsinchu.

The Hsinchu facility was affected by an earthquake.

Current estimated production delay is 14 days.
"""

document_data = {
    "document": "DOC001",
    "title": "Apex Motion's Pune plant manufactures BMS Controller X1.",
    "source": "Apex Motion's Pune plant",
    "text": document,
}


In [ ]:
def run_query(query, parameters=None):
    records, _, _ = driver.execute_query(query, parameters_=parameters or {})
    return [record.data() for record in records]


In [ ]:
query = """
CREATE (d:Document {document: $document, title: $title, source: $source, text: $text})
"""
run_query(query, parameters=document_data)


In [ ]:
chunks = [
    {
        "chunk_id": "C001",
        "text": "Apex Motion's Pune plant manufactures BMS Controller X1."
    },
    {
        "chunk_id": "C002",
        "text": "BMS Controller X1 requires MCU-900."
    },
    {
        "chunk_id": "C003",
        "text": "MCU-900 is supplied by Shenzhen Microelectronics."
    },
    {
        "chunk_id": "C004",
        "text": "Shenzhen Microelectronics depends on Taiwan Silicon Corp for semiconductor wafers."
    },
    {
        "chunk_id": "C005",
        "text": "Taiwan Silicon Corp operates a facility in Hsinchu."
    },
    {
        "chunk_id": "C006",
        "text": "The Hsinchu facility was affected by an earthquake. Current estimated production delay is 14 days."
    }
]

query = """
CREATE (c:Chunk {chunk_id: $chunk_id, text: $text})
"""
for chunk in chunks:
    run_query(query, parameters=chunk)


In [ ]:
query = """
MATCH (d:Document {document: $document})
MATCH (c:Chunk {chunk_id: $chunk_id})
MERGE (d)-[:HAS_CHUNK]->(c)
"""
for chunk in chunks:
    run_query(query, parameters={"document": "DOC001", "chunk_id": chunk["chunk_id"]})
